In [26]:
import torch
import torch.nn as nn
import torch.optim as optim

# 定义RNN类

In [27]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        
        self.W_i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_i2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)
        
    def forward(self, input, hidden):
        combined = torch.cat((input, hidden), 1)
        hidden = self.W_i2h(combined)
        output = self.W_i2o(hidden)
        output = self.softmax(output)
        
        return output, hidden
    
    def init_hidden(self):
        # 初始化隐藏层
        return torch.zeros(1, self.hidden_size)
    

# 创建数据和超参数
设定 RNN 的输入、隐藏层大小、输出大小等超参数。这里用简单的随机数据进行演示。

In [28]:
input_size = 5   # 输入的特征维度
hidden_size = 10 # 隐藏层的特征维度
output_size = 2  # 输出的特征维度（例如二分类问题）

rnn = SimpleRNN(input_size, hidden_size, output_size)

# 创建随机数据：例如一个序列数据，包含 3 个时间步
sequence = [torch.randn(1, input_size) for _ in range(3)]  # 3 时间步的序列
target = torch.tensor([1])  # 假设目标输出是分类标签 1

# 定义损失函数和优化器
使用交叉熵损失函数以及 SGD 优化器。

In [29]:
criterion = nn.NLLLoss()
optimizer = optim.SGD(rnn.parameters(), lr=0.01)

# 训练循环
定义一个简单的训练循环。

In [30]:
num_epoches = 100
for epoch in range(num_epoches):
    rnn.zero_grad()   # 清除前一轮的梯度
    hidden = rnn.init_hidden()  # 初始化隐藏层
    
    for input in sequence:
        output, hidden = rnn(input, hidden)
        
    # 计算损失
    loss = criterion(output, target)
    
    # 反向传播
    loss.backward()
    optimizer.step() 
    
    # 打印损失值
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

Epoch 0, Loss: 0.5717955231666565
Epoch 10, Loss: 0.38658663630485535
Epoch 20, Loss: 0.26946255564689636
Epoch 30, Loss: 0.19348056614398956
Epoch 40, Loss: 0.14358536899089813
Epoch 50, Loss: 0.1101246029138565
Epoch 60, Loss: 0.08703982084989548
Epoch 70, Loss: 0.07061872631311417
Epoch 80, Loss: 0.058587294071912766
Epoch 90, Loss: 0.0495307594537735


# 解释说明
1. 数据输入：在训练过程中，模型接收一个序列，每一个时间步的数据经过前向传播，将输出与更新后的隐藏状态返回。
2. 损失计算：只在序列的最后一个时间步计算损失。对于“许多对一”模型，通常选择最后一个时间步的输出与真实标签进行对比。
3. 梯度更新：反向传播的梯度计算和参数更新，通过 loss.backward() 和 optimizer.step() 完成。

# 直接使用RNN模型

In [31]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(SimpleRNN, self).__init__()
        
        # 定义RNN层
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        
        # 定义全连接层，用于将RNN的输出映射到目标输出
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # 初始化隐藏状态
        h0 = torch.zeros(num_layers, x.size(0), hidden_size)
        
        # 将输入x传入RNN
        out, hn = self.rnn(x, h0)
        
        # 取最后一个时间步的输出，传入全连接层
        out = self.fc(out[:, -1, :])
        
        return out

In [32]:
input_size = 10      # 输入特征的维度
hidden_size = 20     # 隐藏层的维度
output_size = 1      # 输出的维度（例如二分类任务中为1）
num_layers = 2       # RNN的层数

In [33]:
model = SimpleRNN(input_size, hidden_size, output_size, num_layers)

In [34]:
batch_size = 5
sequence_length = 7
x = torch.randn(batch_size, sequence_length, input_size)

# 前向传播
output = model(x)

print("Output shape:", output.shape)

Output shape: torch.Size([5, 1])


# 解释各部分含义
- input_size：每个时间步输入的特征维度。
- hidden_size：隐藏层的维度，决定了隐藏状态的大小。
- num_layers：RNN的层数。
- output_size：模型输出的维度，常用于映射为实际需要的输出格式。
在这个例子中，模型先将输入传入RNN层得到时间序列的输出，然后选取最后一个时间步的输出，经过全连接层后输出最终结果。
# 常见操作和技巧
- 初始化隐藏状态：通常在每次前向传播中都需要初始化隐藏状态h0，尤其是在处理独立的数据批次时。
- 选择时间步的输出：在很多任务中，我们只关心最后一个时间步的输出（例如句子分类任务），但也可以取所有时间步的输出用于不同任务（如机器翻译）。
- 多层RNN：增加num_layers可以创建深层RNN，捕捉更复杂的模式。